In [0]:
from datetime import datetime
from pyspark.sql.functions import max as spark_max

# Optimize shuffle for small clusters
spark.conf.set("spark.sql.shuffle.partitions", 1)

# 1. Widgets & Setup
dbutils.widgets.text("customer_id", "cust_001")
dbutils.widgets.text("object_name", "events")
dbutils.widgets.text("source_system", "bigquery")
dbutils.widgets.text("bucket_path", "s3://hgi-databricks-data-lakehouse-dev/")
dbutils.widgets.text("bq_catalog_table", "`bigquery-connection_catalog`.`v4c_bigquery_dataset`.`event_raw`")

customer_id = dbutils.widgets.get("customer_id")
object_name = dbutils.widgets.get("object_name")
source_system = dbutils.widgets.get("source_system")
bucket_path = dbutils.widgets.get("bucket_path")
bq_catalog_table = dbutils.widgets.get("bq_catalog_table")

# Timestamp path
now = datetime.now()
yyyy, mm, dd, hh = now.strftime("%Y"), now.strftime("%m"), now.strftime("%d"), now.strftime("%H")
incremental_path = f"{bucket_path}landing-zone/{source_system}/{customer_id}/{object_name}/incremental/{yyyy}/{mm}/{dd}/{hh}"

print(f"🚀 Starting Incremental Load to: {incremental_path}")

# 2. Get the last watermark
try:
    watermark_df = spark.sql(f"""
    SELECT last_processed_timestamp
    FROM ingestion_metadata.watermarks
    WHERE source_system='{source_system}'
    AND object_name='{object_name}'
    """)
    rows = watermark_df.collect()
    if len(rows) == 0:
        raise Exception("Watermark missing.")
    watermark = rows[0][0]
except Exception as e:
    raise Exception(f"❌ Watermark not initialized. Please run historical load first. Error: {str(e)}")

print(f"📍 Last Watermark: {watermark}")

# 3. Query BigQuery FIRST to get the NEW max timestamp 
new_ts_df = spark.sql(f"""
SELECT MAX(event_timestamp) as max_ts
FROM {bq_catalog_table}
WHERE event_timestamp > TIMESTAMP('{watermark}')
""")

new_ts = new_ts_df.first()[0]

if new_ts is None:
    print("✅ No new records found in BigQuery. Exiting gracefully.")
    dbutils.notebook.exit("0")

print(f"🎯 New records found up to: {new_ts}")

# 4. Pull the data using BOTH bounds and write it
df_incremental = spark.sql(f"""
SELECT *
FROM {bq_catalog_table}
WHERE event_timestamp > TIMESTAMP('{watermark}')
  AND event_timestamp <= TIMESTAMP('{new_ts}')
""")

try:
    # 5. Write to S3 
    df_incremental.write \
        .mode("append") \
        .format("parquet") \
        .option("compression", "snappy") \
        .save(incremental_path)
    print("💾 Write to S3 completed successfully.")
    
except Exception as e:
    print(f"❌ Failed during write operation: {str(e)}")
    raise e

# 6. Update Watermark with 1-Minute Lookback
print("Updating watermark table with lookback window...")

# Notice the `- INTERVAL 1 MINUTE` added here
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW new_watermarks AS
    SELECT source_system, object_name, 
           CASE 
               WHEN source_system = '{source_system}' AND object_name = '{object_name}' THEN TIMESTAMP('{new_ts}') - INTERVAL 1 MINUTE
               ELSE last_processed_timestamp 
           END as last_processed_timestamp
    FROM ingestion_metadata.watermarks
""")

spark.sql("INSERT OVERWRITE TABLE ingestion_metadata.watermarks SELECT * FROM new_watermarks")

print(f"✅ Watermark updated successfully (minus 1 minute).")
dbutils.notebook.exit("Success")